# AirfRANS GNN Surrogate — Colab GPU setup

This notebook is infrastructure only: get the repo, the dependencies, and the
dataset onto a real GPU. Full-resolution mesh graphs (~180k nodes / ~720k edges
per case) hung the local dev machine even 5 at a time (CPU-only, 4GB GPU) — that's
exactly why real training doesn't happen locally. All real logic still lives in
`src/`; this notebook just proves the environment works, it doesn't contain
training logic itself.

Runtime > Change runtime type > GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo

Colab VMs reset every session, so the repo needs to come from somewhere durable.
Push this repo to GitHub first, then replace the URL below. (Alternative: mount
Google Drive and keep the repo there instead — swap this cell for a Drive mount
if you'd rather not use GitHub.)

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo

## 2. Install dependencies

Colab ships torch with CUDA already installed — don't reinstall it. `torch_geometric`
installs as pure Python (no `torch-scatter`/`torch-sparse` needed; our model only
uses `torch_geometric.utils.scatter`, confirmed working locally).

In [ ]:
!pip install -q torch_geometric lightning airfrans pyvista

## 3. Get the dataset

Raw dataset (~15GB) stays on Colab's local disk -- it fits fine alone, and
Google Drive's free tier (15GB total, shared with Gmail/Photos) can't hold
both the raw dataset *and* the cache. Only the much smaller preprocessed
cache (~9.6GB, section 5) goes to Drive, since that's the expensive-to-recompute
part worth protecting across session resets. The raw download itself is a
"wait a bit" cost if a session resets -- re-parsing 800 VTU files is not.</cell id="cell-6">


In [ ]:
import os
import airfrans as af
from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT = "data"  # raw dataset -- LOCAL disk, ephemeral, re-downloaded if a session resets
DRIVE_ROOT = "/content/drive/MyDrive/airfrans-gnn-data"  # cache + checkpoint -- persist here
STATS_PATH = "data/norm_stats.npz"  # small, tracked in git, stays in the repo checkout

os.makedirs(DRIVE_ROOT, exist_ok=True)
dataset_root = os.path.join(DATA_ROOT, "Dataset")

if not os.path.isdir(dataset_root) or not os.listdir(dataset_root):
    af.dataset.download(root=DATA_ROOT, file_name="Dataset", unzip=True, OpenFOAM=False)

print("manifest present:", os.path.isfile(os.path.join(dataset_root, "manifest.json")))

## 4. Smoke test: one full-resolution case on the real GPU

Same forward/backward timing check done locally on CPU (11s/case there) — here
it should be dramatically faster, and use the *real* mesh graph
(`build_graph`), not the local k-NN subsample workaround
(`build_subsampled_graph`, dev-machine-only, see `src/graph.py`).

In [ ]:
import time
import torch

from src.data import split_names
from src.dataset import PyGAirfRANSDataset
from src.model import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train_names = split_names(dataset_root, task="full", train=True)[:1]
ds = PyGAirfRANSDataset(dataset_root, train_names)  # no stats yet -- just a smoke test
data = ds[0].to(device)

model = MeshGraphNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
pred = model(data.x, data.edge_index, data.edge_attr)
loss = torch.nn.functional.mse_loss(pred, data.y)
loss.backward()
opt.step()
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}, "
      f"forward+backward+step={time.time()-t0:.2f}s, loss={loss.item():.4f}")

## 5. Preprocess: cache real mesh graphs to Drive

One-time cost per case (VTU parsing dominates, not GPU-bound). Skips any case
already cached, so it's safe to re-run/resume after a disconnect. `CACHE_DIR`
is under `DRIVE_ROOT`, not the local dataset -- so once this fully completes,
future sessions find it already there even after re-downloading the raw
dataset fresh.

In [ ]:
from src.data import split_names
from src.preprocess import preprocess_split

CACHE_DIR = os.path.join(DRIVE_ROOT, "cache", "full")
train_names = split_names(dataset_root, task="full", train=True)

preprocess_split(dataset_root, train_names, CACHE_DIR)
print(f"cached {len(train_names)} training cases to {CACHE_DIR}")

## 5b. Copy the cache to local disk for training

Reading every cached case from Drive's network mount on every single step was
the dominant cost behind ~19s/step earlier (GPU compute for this tiny model
should be well under 1s/step). The raw dataset is no longer needed locally now
that preprocessing is done -- deleting it frees room to copy the much smaller
cache down from Drive for fast local reads during training. Drive still holds
the source of truth; this local copy is disposable.

In [ ]:
import shutil

LOCAL_CACHE_DIR = "local_cache"
# train.py's split_names() only ever reads manifest.json -- keep just that one
# small file locally so deleting the raw dataset doesn't break it.
MANIFEST_DIR = "local_manifest"

os.makedirs(MANIFEST_DIR, exist_ok=True)
shutil.copy(os.path.join(dataset_root, "manifest.json"), os.path.join(MANIFEST_DIR, "manifest.json"))

shutil.rmtree(dataset_root, ignore_errors=True)  # raw VTU -- training only reads the cache

if not os.path.isdir(LOCAL_CACHE_DIR) or len(os.listdir(LOCAL_CACHE_DIR)) < len(train_names):
    shutil.copytree(CACHE_DIR, LOCAL_CACHE_DIR, dirs_exist_ok=True)

print(f"local cache: {len(os.listdir(LOCAL_CACHE_DIR))}/{len(train_names)} cases")

## 6. Train

Trains on the 800 official training cases minus a validation holdout (`n_val`),
reporting relative L2 per field (pressure, vx, vy, nu_t separately -- never one
blended number, see CLAUDE.md) on validation each epoch. The official 200-case
test split is deliberately untouched here -- it's reserved for week 5's lift/drag
evaluation, not used for training-time monitoring.

`batch_size=4` OOM'd a 16GB T4 (batching full-resolution graphs is expensive --
~2.88M edges x 4 message-passing rounds of retained activations). `batch_size=1`
with `accumulate_grad_batches=4` gets the same effective batch size for gradient
statistics without holding more than one graph in memory at a time.

`precision="16-mixed"` uses the T4's tensor cores and halves activation memory
(more headroom against another OOM). Reads from `LOCAL_CACHE_DIR` (section 5b),
not Drive directly, to avoid the ~19s/step network-read bottleneck.

In [ ]:
from src.train import main as train_main

train_main(
    dataset_root=MANIFEST_DIR,  # only manifest.json is needed here (see section 5b)
    cache_dir=LOCAL_CACHE_DIR,  # local copy, not Drive -- fast reads
    stats_path=STATS_PATH,
    checkpoint_path=os.path.join(DRIVE_ROOT, "meshgraphnet.ckpt"),  # Drive -- survives resets
    max_epochs=100,
    batch_size=1,
    accumulate_grad_batches=4,
    n_val=80,
    checkpoint_every_n_epochs=5,  # periodic checkpoints land in DRIVE_ROOT too
    num_workers=2,
    precision="16-mixed",
)